In [1]:
import os
import re
import pandas as pd

def parse_restraint(restraint):
    try:
        if not isinstance(restraint, str):
            return None, None, None, None

        restraint = restraint.strip()
        if restraint.count('-') != 1:
            return None, None, None, None

        part1, part2 = restraint.split('-')

        def split_part(part):
            match = re.match(r'^(\d+)(.+)', part)
            if match:
                return match.groups()
            else:
                return None, part

        res1, atom1 = split_part(part1)
        res2, atom2 = split_part(part2)

        if res2 is None:
            res2 = res1

        return res1, atom1, res2, atom2

    except Exception as e:
        print(f"Error: {e}")
        return None, None, None, None

def convert_single_restraint_csv_to_itp(csv_file, itp_path, mol_name, outname=None, print_restraint=False):
    from pprint import pprint 

    ### Parsed restraints from csv file 

    df = pd.read_csv(csv_file)
    df.columns = ['restraint', 'intensity']

    print('row extration restraints from csv', df)
    intensity_priority = {'weak': 0, 'medium': 1, 'strong': 2} # Make sure the weeker take higher priority
    bond_map = {
        'strong': '10  0.00 0.25 0.35 5000  0.00 0.25 0.35 0',
        'medium': '10  0.00 0.35 0.45 5000  0.00 0.35 0.45 0',
        'weak':   '10  0.00 0.50 0.60 5000  0.00 0.50 0.60 0'
    }

    cgnr_lookup = {
        'H02': {
            'α11': ['H12', 'H13'],
            'α2': ['H3'],
            'β3': ['H4', 'H5', 'H6'],
            'NH+1':['H1', 'H2']
        },
        'R02': {
            'α11': ['H10', 'H11'],
            'α2': ['H1'],
            'β3': ['H2', 'H3', 'H4']
        },
        'NH2': {
            'NH1': ['H1', 'H2']
        }
    }

    ### Clean the restriants 
    parsed_restraints = []
    for i, row in df.iterrows():
        restraint = row['restraint']
        intensity = row['intensity']

        if pd.isna(restraint) or pd.isna(intensity):
            continue

        res1, atom1, res2, atom2 = parse_restraint(str(restraint))

        if None not in (res1, atom1, res2, atom2):
            parsed_restraints.append({
                'res1': res1,
                'atom1': atom1.replace('α12', 'α11').replace('NH2', 'NH1').replace('NH+2', 'NH+1'),
                'res2': res2,
                'atom2': atom2.replace('α12', 'α11').replace('NH2', 'NH1').replace('NH+2', 'NH+1'),
                'intensity': intensity
            })

    df_parsed = pd.DataFrame(parsed_restraints)
    print('Change the name of restraints:', df_parsed)
    df_parsed['intensity_score'] = df_parsed['intensity'].map(intensity_priority)
    #print('Update the priority of the restraints:', df_parsed)
    df_sorted = df_parsed.sort_values('intensity_score')
    #print('df_sorted:', df_sorted)

    df_merged = df_sorted.drop_duplicates(subset=['res1', 'atom1', 'res2', 'atom2'], keep='first')
    #print('df_merged:', df_merged)
    itp_file = f"{mol_name}.itp"
    atoms_file = os.path.join(itp_path, itp_file)

    if not os.path.exists(atoms_file):
        print(f"❌ Missing .itp file for {mol_name}: {atoms_file}")
        return



    # Parse atom section from .itp
    start_tag = "[ atoms ]"
    block = []
    in_block = False

    with open(atoms_file, 'r') as file:
        for line in file:
            stripped = line.strip()
            if stripped.startswith('[') and stripped.endswith(']'):
                if stripped.lower() == start_tag:
                    in_block = True
                    continue
                elif in_block:
                    break
            if in_block and stripped and not stripped.startswith(';'):
                block.append(line.strip())

    atom_records = []
    for line in block:
        line = line.split(';')[0].strip()
        parts = line.split()
        if len(parts) >= 6:
            nr, atom_type, resi, res, atom, cgnr = parts[:6]
            atom_records.append([nr, atom_type, resi, res, atom, cgnr])

    df_atoms = pd.DataFrame(atom_records, columns=['nr', 'type', 'resi', 'res', 'atom', 'cgnr'])

    print('df_atoms', df_atoms)


    ### Create the restraints list 
    restraint_list = []
    j = 0
    for i, row in df_merged.iterrows():
        res1_i = int(row['res1'])
        res2_i = int(row['res2'])
        atom1 = row['atom1']
        atom2 = row['atom2']
        intensity = row['intensity']
        
        if atom1 == 'NH1':
            res1_i += 1
        if atom2 == 'NH1':
            res2_i += 1
        print('res1_i', res1_i, 'res2_i', res2_i, 'atom1', atom1, 'atom2', atom2)

            
        res1_i = str(res1_i)
        res2_i = str(res2_i)

        match1 = df_atoms[df_atoms['resi'] == res1_i]
        match2 = df_atoms[df_atoms['resi'] == res2_i]

        if match1.empty or match2.empty:
            continue

        res1_res_name = match1['res'].values[0]
        res2_res_name = match2['res'].values[0]

        cgnr_labels1 = cgnr_lookup.get(res1_res_name, {}).get(atom1, [])
        cgnr_labels2 = cgnr_lookup.get(res2_res_name, {}).get(atom2, [])

        cgnr_nums_1 = df_atoms[(df_atoms['resi'] == res1_i) & (df_atoms['atom'].isin(cgnr_labels1))]['cgnr'].tolist()
        cgnr_nums_2 = df_atoms[(df_atoms['resi'] == res2_i) & (df_atoms['atom'].isin(cgnr_labels2))]['cgnr'].tolist()
        print('cgnr_nums_1', cgnr_nums_1, 'cgnr_nums_2', cgnr_nums_2)
        print(f"from {j}, to {j + len(cgnr_nums_1) * len(cgnr_nums_2) - 1}")
        j += int(len(cgnr_nums_1)) * int(len(cgnr_nums_2))
        
        for c1 in cgnr_nums_1:
            for c2 in cgnr_nums_2:
                restraint_list.append({
                    'atom1': c1,
                    'atom2': c2,
                    'Intensity': intensity
                })

    df_ready = pd.DataFrame(restraint_list)

    restraint_lines = []
    restraints = []
    restraint_lines.append(";  ai   aj  funct  r0_A  r1_A  r2_A  k_A  r0_B  r1_B  r2_B  k_B  flat-bottom parameters")
    for i, row in df_ready.iterrows():
        ai = row['atom1']
        aj = row['atom2']
        intensity = row['Intensity'].lower()
        bond = bond_map.get(intensity)
        if bond:
            restraint_line = f"  {ai}  {aj}  {bond}"
            restraint = [f"{ai}",f"{aj}", f"{intensity}"]
            restraint_lines.append(restraint_line)
            restraints.append(restraint)

    if print_restraint:
        df_restraints = pd.DataFrame(restraints)
        file_name = f"{mol_name}_restraints.csv"
        df_restraints.to_csv(file_name, index=False)
        print(f"Save the restrians list to {file_name}")
        pprint(df_restraints)
        

    # Insert into .itp
    start_tag = '[ bonds ]'
    new_itp_file = itp_file.replace('.itp', f'{outname}_re.itp')
    new_itp_path = os.path.join(itp_path, new_itp_file)

    with open(atoms_file, 'r') as f:
        lines = f.readlines()

    output_lines = []
    in_bonds_section = False
    bonds_inserted = False

    for line in lines:
        stripped = line.strip()
        output_lines.append(line)

        if stripped == start_tag:
            in_bonds_section = True
            continue

        if in_bonds_section and stripped.startswith('[') and stripped.endswith(']'):
            if not bonds_inserted:
                output_lines = output_lines[:-1]
                output_lines.extend([r + '\n' for r in restraint_lines])
                output_lines.append(line)
                bonds_inserted = True
            in_bonds_section = False
            continue

    if in_bonds_section and not bonds_inserted:
        output_lines[-1] = output_lines[-1].rstrip('\n')
        output_lines.extend([r + '\n' for r in restraint_lines])

    with open(new_itp_path, 'w') as f:
        f.writelines(output_lines)

    print(f"✅ Appended restraints to [ bonds ] section and saved as: {new_itp_path}")


In [3]:
convert_single_restraint_csv_to_itp('restraints_file/nspe_10_res.csv', 'itp', 'nspe_10', outname='',  print_restraint=True)

row extration restraints from csv      restraint intensity
0   10α11-1α12    strong
1     5α11-1α2    medium
2      7β3-1β3    medium
3      5β3-1β3    medium
4     3β3-1α11    medium
5   8α12-1NH+1    medium
6   8α12-1NH+2    medium
7   6α12-1NH+2    medium
8   6α12-1NH+1      weak
9    1α11-2α12    medium
10   2α11-3α12    strong
11   3α12-2α12    medium
12    4β3-3α11    medium
13    4β3-3α12    medium
14    3α11-4α2    strong
15    3α12-4α2    strong
16    4α11-5α2    strong
17    6β3-5α11    strong
18    5α12-6α2    strong
19   6α11-7α11    strong
20    7α11-8α2    strong
21    7α12-8α2    strong
22   8α11-9α11    strong
23   9α11-10α2    strong
24   9α12-10α2    strong
25     1β3-α11    strong
26     1β3-α12    strong
27     1α11-α2    strong
28     2β3-α11    strong
29     3β3-α11    strong
30     4β3-α11    strong
31     5β3-α11    strong
32     6β3-α11    strong
33     7β3-α12    strong
34     8β3-α11    strong
35     9β3-α12    strong
36    10β3-α11    strong
37    10α11-α2  

In [8]:
convert_single_restraint_csv_to_itp('restraints_file/nspe_7_1_res.csv', 'itp', 'nspe_7_1', outname='_5k',  print_restraint=True)
convert_single_restraint_csv_to_itp('restraints_file/nspe_7_2_res.csv', 'itp', 'nspe_7_2', outname='_5k',  print_restraint=True)
convert_single_restraint_csv_to_itp('restraints_file/nspe_10_res.csv', 'itp', 'nspe_10', outname='_5k',  print_restraint=True)

row extration restraints from csv     restraint intensity
0   6α12-7NH2    medium
1   6α12-7NH1      weak
2   3α12-7NH1      weak
3    1α11-2α2    strong
4    1α12-2α2    strong
5   3α12-2α12    strong
6   4α11-3α11    strong
7    5β3-4α11    medium
8    4α11-5α2    strong
9    4α12-5α2    strong
10   5α11-6α2    strong
11   5α12-6α2    strong
12   6β3-5α11    strong
13  6α11-7α12    strong
14  7α12-6α12    strong
15    2β3-α11    strong
16    2α11-α2    medium
17    3β3-α11    strong
18    3α11-α2    medium
19    6β3-α11    strong
20    7β3-α11    strong
21    7α11-α2    medium
22   7α12-NH2      weak
23   7α12-NH1      weak
df_atoms       nr type resi  res atom cgnr
0      1    n    1  H02    N    1
1      2   c3    1  H02   C1    2
2      3   c3    1  H02   C2    3
3      4   ca    1  H02   C3    4
4      5   ca    1  H02   C4    5
..   ...  ...  ...  ...  ...  ...
161  162    c    7  R02    C  162
162  163    o    7  R02    O  163
163  164    n    8  NH2    N  164
164  165   hn    